# Class 3: Shiller data and an Altair chart

Rajwardhan Rana · Data Visualisation for Policymaking

Adapted from the supplied DS3_2_Requests_Shiller and DS3_3_Altair_Shiller notebooks by the LSE PP434 teaching team. Original materials are in the Class3_materials folder alongside the project.

Run one cell at a time with Shift + Enter. Read the explanation and inspect the output before moving on. In Colab, first choose File → Save a copy in Drive. The website uses the supplied teaching snapshot, not live market data.

In [1]:
# Run before importing packages in Colab. Local requirements are in scripts/requirements-week3.txt.
import importlib.util
import subprocess
import sys
needed = [name for name in ['requests', 'pandas', 'altair'] if importlib.util.find_spec(name) is None]
if needed:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *needed])


## 1. Fetch data with requests
Read Date as text so 1871.10 remains October, rather than becoming 1871.1 (January). A SHA-256 check keeps the published snapshot reproducible.

In [2]:
from pathlib import Path
import hashlib
import io
import requests
import pandas as pd
import altair as alt

# Works from the project root, its notebooks folder, or a fresh Colab session.
root = Path.cwd()
if root.name in ('notebooks', 'scripts') and (root.parent / 'index.html').exists():
    root = root.parent
data_dir = root / 'data'
data_dir.mkdir(exist_ok=True)
csv_url = 'https://raw.githubusercontent.com/jhellingsdata/LSE-PP434/main/weeks/week03/data/data_shiller.csv'
response = requests.get(csv_url, timeout=60)
response.raise_for_status()
expected_hash = 'd77861ce8e5981ebb3a916b197fcf372a7dea0b15c709a44d1d4b639891f10e2'
if hashlib.sha256(response.content).hexdigest() != expected_hash:
    # The teaching source may have been updated: retain our checked snapshot.
    snapshot_url = 'https://raw.githubusercontent.com/RajwardhanRana/RajwardhanRana.github.io/main/data/data_shiller.csv'
    response = requests.get(snapshot_url, timeout=60)
    response.raise_for_status()
    if hashlib.sha256(response.content).hexdigest() != expected_hash:
        raise ValueError('Source snapshot changed. Review the data before regenerating charts.')
(data_dir / 'data_shiller.csv').write_bytes(response.content)
df = pd.read_csv(io.StringIO(response.text), dtype={'Date': str})
print('Raw data shape:', df.shape)
print(df[['Date', 'P', 'E']].head(12).to_string(index=False))


Raw data shape: (1870, 9)
   Date    P   E
1871.01 4.44 0.4
1871.02 4.50 0.4
1871.03 4.61 0.4
1871.04 4.74 0.4
1871.05 4.86 0.4
1871.06 4.82 0.4
1871.07 4.73 0.4
1871.08 4.79 0.4
1871.09 4.84 0.4
1871.10 4.59 0.4
1871.11 4.64 0.4
1871.12 4.74 0.4


## 2. Calculate P/E and remove missing observations
P/E is price divided by earnings (P/E). Keep the original table intact. Remove only rows without a finite ratio; missing CAPE values in the first decade do not invalidate ordinary P/E.

In [3]:
import numpy as np
df['PE_ratio'] = df['P'] / df['E']
usable = np.isfinite(df['PE_ratio'])
df_clean = df.loc[usable].copy()
print('Rows removed for missing or non-finite P/E:', int((~usable).sum()))
print('Cleaned rows:', len(df_clean))
assert np.isfinite(df_clean['PE_ratio']).all()


Rows removed for missing or non-finite P/E: 4
Cleaned rows: 1866


## 3. Parse and check monthly dates
Dates are the first day of each month as YYYY-MM-DD. Check duplicate dates and October explicitly. Export both the full cleaned table and a small two-column file.

In [4]:
df_clean['Date_iso'] = pd.to_datetime(df_clean['Date'], format='%Y.%m', errors='raise')
df_clean = df_clean.sort_values('Date_iso').reset_index(drop=True)
assert not df_clean['Date_iso'].duplicated().any()
assert df_clean.loc[df_clean['Date'].eq('1871.10'), 'Date_iso'].iloc[0] == pd.Timestamp('1871-10-01')
assert len(pd.period_range(df_clean.Date_iso.min(), df_clean.Date_iso.max(), freq='M')) == len(df_clean)
df_clean.to_csv(data_dir / 'data_shiller_clean.csv', index=False, date_format='%Y-%m-%d')
df_out = df_clean[['Date_iso', 'PE_ratio']].copy()
df_out.to_csv(data_dir / 'data_shiller_pe.csv', index=False, date_format='%Y-%m-%d')
print(df_out.head(12).to_string(index=False))
print('Period:', df_out.Date_iso.min().date(), 'to', df_out.Date_iso.max().date())


  Date_iso  PE_ratio
1871-01-01    11.100
1871-02-01    11.250
1871-03-01    11.525
1871-04-01    11.850
1871-05-01    12.150
1871-06-01    12.050
1871-07-01    11.825
1871-08-01    11.975
1871-09-01    12.100
1871-10-01    11.475
1871-11-01    11.600
1871-12-01    11.850
Period: 1871-01-01 to 2026-06-01


## 4. Inspect the results
The average is the arithmetic mean of all valid monthly ratios in this snapshot. It is a descriptive reference, not an estimate of fair value. The latest observation is the latest usable month in the teaching file, not today’s market valuation.

In [5]:
mean_pe = float(df_out.PE_ratio.mean())
latest = df_out.iloc[-1]
peak = df_out.loc[df_out.PE_ratio.idxmax()]
print(f'Mean monthly P/E: {mean_pe:.2f}')
print(f'Latest usable observation: {latest.Date_iso:%B %Y}, P/E {latest.PE_ratio:.2f}')
print(f'Highest observed P/E: {peak.PE_ratio:.2f}, {peak.Date_iso:%B %Y}')


Mean monthly P/E: 16.22
Latest usable observation: June 2026, P/E 25.22
Highest observed P/E: 123.73, May 2009


## 5. Build and style the chart in Altair
Altair expresses Vega-Lite in Python: data → mark → encoding. Compared with the handout, this changes the colour, line width, title, size, axis titles and font sizes, and adds hover details. The optional dashed average line uses layering (`+`).

In [6]:
def build_chart(source):
    series = alt.Chart(source).mark_line(color='#087f8c', strokeWidth=2).encode(
        x=alt.X('Date_iso:T', title='Year', axis=alt.Axis(grid=False, format='%Y', tickCount=7)),
        y=alt.Y('PE_ratio:Q', title='Price / earnings (times)', scale=alt.Scale(zero=True)),
        tooltip=[alt.Tooltip('Date_iso:T', title='Month', format='%b %Y'),
                 alt.Tooltip('PE_ratio:Q', title='P/E (times)', format='.2f')]
    )
    average = alt.Chart(source).mark_rule(color='#a56316', strokeDash=[6, 4], strokeWidth=1.5).encode(
        y=alt.Y('mean(PE_ratio):Q', title='Price / earnings (times)'),
        tooltip=[alt.Tooltip('mean(PE_ratio):Q', title='Sample mean P/E', format='.2f')]
    )
    return (series + average).properties(
        width='container', height=320,
        title=alt.Title(text='US stock prices relative to earnings',
                        subtitle=['Monthly S&P composite P/E; dashed line = sample mean',
                                  'Source: Robert Shiller, via the course data snapshot'],
                        anchor='start', fontSize=18, subtitleFontSize=11)
    ).configure_view(stroke=None).configure_axis(
        labelFontSize=12, titleFontSize=13, titlePadding=12,
        gridColor='#e8edf0', domainColor='#b8c4cc'
    ).configure(font='Arial', background='white')

# A DataFrame embeds values and is convenient for the notebook preview.
chart = build_chart(df_out)
chart


Altair chart: monthly P/E with sample average

## 6. Export a small JSON for the website
Use a URL instead of a DataFrame to keep data out of the JSON. This relative CSV URL works when the chart is embedded in the root index.html. The CSV and JSON must be published together. To try the exported spec directly in the Vega Editor, change the data URL to the raw GitHub URL printed below.

In [7]:
web_chart = build_chart('data/data_shiller_pe.csv')
web_chart.save(str(root / 'wk3_pe_ratio_chart.json'))
spec = web_chart.to_dict(validate=True)
assert spec['data']['url'] == 'data/data_shiller_pe.csv'
assert 'datasets' not in spec
print('Saved:', root / 'wk3_pe_ratio_chart.json')
print('Raw data URL: https://raw.githubusercontent.com/RajwardhanRana/RajwardhanRana.github.io/main/data/data_shiller_pe.csv')


Saved: D:\Data Visualization for policy making\RajwardhanRana.github.io\wk3_pe_ratio_chart.json
Raw data URL: https://raw.githubusercontent.com/RajwardhanRana/RajwardhanRana.github.io/main/data/data_shiller_pe.csv


## Interpretation and limitations
This is ordinary P/E calculated as P divided by E. It is not CAPE, which uses smoothed inflation-adjusted earnings. The large 2009 spike reflects the denominator problem discussed in the course: earnings fell very sharply, so the ratio rose. Being above a historical average does not by itself prove overvaluation.

The snapshot is static; rerunning the notebook reproduces the supplied data rather than refreshing live market prices.

## Practice
Use the original handout notebooks to practise Python basics (DS3_4), Altair (DS3_5), pandas (DS3_6), requests (DS3_7) and the optional original-Excel workflow (DS3_8). Their exercises have not been marked as completed for you.

## Optional Colab downloads
In Colab, open Files → data and download both cleaned CSVs, then download wk3_pe_ratio_chart.json from the main Files folder. No Google sign-in or Drive changes were performed by this local build.